# Laborator

## Regresie Liniara

### Cerință

Estimăm durata totală a unei livrări în funcție de distanță, pentru aceeași zonă de livrare. Durata include pregătirea și deplasarea.

Date:
- Distanță (km): `[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]`
- Durată (minute): `[12, 16, 17, 22, 23, 27, 29, 30, 35, 36, 40, 42]`

Sarcini:
1. Vizualizarea datelor pentru a înțelege natura relației.
2. Antrenarea unui model de regresie liniară simplă.
3. Calcularea metricilor de evaluare: MAE, MSE, RMSE și R<sup>2</sup>.
4. Predicție pentru o livrare la 7,5 km folosind modelul antrenat.
5. Discuție despre adecvarea modelului liniar (underfitting) și validitatea predicției.

### Rezolvare

Importăm bibliotecile necesare.

In [ ]:
import matplotlib.pyplot as plt  # pentru grafice
import numpy as np  # pentru calcul algebric
from sklearn.linear_model import LinearRegression  # algoritmul de regresie liniară
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score  # metrici de evaluare

Definim datele cu care vom lucra (din enunțul problemei).

In [ ]:
distance_km = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]).reshape(-1, 1)
duration_min = np.array([12, 16, 17, 22, 23, 27, 29, 30, 35, 36, 40, 42])

Trasăm grafic punctele din textul problemei pentru vizualizare.

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(distance_km, duration_min, color='blue', label='Data points')
plt.xlabel('Distance (km)')
plt.ylabel('Delivery time (min)')
plt.title('Delivery Time vs Distance')
plt.grid(True)
plt.legend()
plt.show()

Creăm și antrenăm modelul.

In [ ]:
model = LinearRegression()
model.fit(distance_km, duration_min)

Afișăm coeficienții rezultați după antrenare.

In [ ]:
coef = model.coef_[0]
intercept = model.intercept_
print(f"Model Coefficients:")
print(f"  Coefficient: {coef:.2f}")
print(f"  Intercept: {intercept:.2f}")

Regresia liniară este descrisă de o funcție de gradul 1 (o dreaptă). După antrenare modelul este $t(d) = 9.98 + 2.68d$, cu $d$ în km și $t$ în minute. Un kilometru suplimentar adaugă aproximativ 2,68 minute; termenul liber aproximează componenta fixă a duratei.

Calculăm metricile pe datele de antrenare. Predicția pentru 7,5 km este o interpolare în intervalul observat.

In [ ]:
predictions = model.predict(distance_km)

mae = mean_absolute_error(duration_min, predictions)
mse = mean_squared_error(duration_min, predictions)
rmse = np.sqrt(mse)
r2 = r2_score(duration_min, predictions)

print("=== Linear Regression Metrics ===")
print(f"MAE: {mae:.2f}")
print(f"MSE: {mse:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.4f}")

plt.figure(figsize=(8, 6))
plt.scatter(distance_km, duration_min, color='blue', label='Data points')
plt.plot(distance_km, predictions, color='red', label='Linear fit')
plt.xlabel('Distance (km)')
plt.ylabel('Delivery time (min)')
plt.title('Linear Regression Fit')
plt.grid(True)
plt.legend()
plt.show()

delivery_distance = np.array([[7.5]])
prediction_delivery = model.predict(delivery_distance)[0]
print(f"\nEstimated delivery time for 7.5 km: {prediction_delivery:.2f} min")


## Regresie logistică

Regresia logistică estimează probabilitatea unei clase; aici folosim pragul 0,5 pentru clasificare.

### Cerință

Dorim să prezicem dacă un student va promova un examen în funcție de orele de studiu.

Date:
- Ore studiate: `[1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9, 9, 10, 10, 10]`
- Promovat (1=Da, 0=Nu): `[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]`

Sarcini:
1. Vizualizarea datelor
2. Antrenarea unui model de regresie logistică
3. Trasarea curbei sigmoid
4. Calcularea matricii de confuzie, acurateții, preciziei, recall-ului și scorului F1
5. Predicție pentru un student care a studiat 4 ore
6. Găsirea orelor de studiu necesare pentru 90% șanse de a promova

### Rezolvare

Importăm bibliotecile necesare.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression  # algoritm de clasificare binară
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score  # metrici de evaluare
import seaborn as sns  # aspect grafice

Definim datele cu care vom lucra (din enunțul problemei).

In [ ]:
hours = np.array([1, 1, 1, 2, 2, 2, 3, 3, 3, 4, 4, 4, 5, 5, 5, 6, 6, 6, 7, 7, 7, 8, 8, 8, 9, 9, 9, 10, 10, 10]).reshape(-1, 1)  # orele de studiu
passed = np.array([0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1])  # promovat: 1 = da, 0 = nu

Afișăm datele. Mărimea punctelor indică numărul de studenți; aceleași ore de studiu pot avea rezultate diferite.

In [ ]:
points, counts = np.unique(np.column_stack((hours.ravel(), passed)), axis=0, return_counts=True)

plt.figure(figsize=(8, 5))
plt.scatter(points[:, 0], points[:, 1], s=60 * counts, color='blue', alpha=0.6, edgecolors='black')
plt.xlabel('Hours studied')
plt.ylabel('Exam outcome')
plt.title('Student Exam Results')
plt.yticks([0, 1], ['Failed', 'Passed'])
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

Creăm și antrenăm modelul.

In [ ]:
model = LogisticRegression()
model.fit(hours, passed)

Afișăm formula rezultată după antrenare și coeficienții acesteia.

In [ ]:
b0 = model.intercept_[0]
b1 = model.coef_[0][0]
print(f"Model Coefficients:")
print(f"  Intercept (b0): {b0:.4f}")
print(f"  Coefficient (b1): {b1:.4f}")
print(f"\nModel equation: P(pass) = 1 / (1 + e^(-({b0:.4f} + {b1:.4f} * hours)))")

Generăm predicțiile și afișăm graficul funcției.

In [ ]:
plt.figure(figsize=(8, 5))

hours_range = np.linspace(1, 10, 300).reshape(-1, 1)
probabilities = model.predict_proba(hours_range)[:, 1]

y_pred = model.predict(hours)

plt.scatter(points[:, 0], points[:, 1], s=60 * counts, color='blue', alpha=0.6, label='Observed outcomes', edgecolors='black', zorder=3)
plt.plot(hours_range, probabilities, color='red', linewidth=2, label='Estimated probability')
plt.axhline(y=0.5, color='green', linestyle='--', label='Probability threshold (0.5)')
plt.xlabel('Hours studied')
plt.ylabel('Probability of passing')
plt.title('Logistic Regression Model')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Generăm matricea de confuzie. `labels=[0, 1]` păstrează forma 2×2 inclusiv pentru un subset cu o singură clasă. Un asemenea subset nu evaluează recunoașterea celeilalte clase.

In [ ]:
cm = confusion_matrix(passed, y_pred, labels=[0, 1])
tn, fp, fn, tp = cm.ravel()

print(f"\n{'=' * 50}")
print("CONFUSION MATRIX")
print(f"{'=' * 50}")
print(f"\n                Predicted")
print(f"              Fail (0)  Pass (1)")
print(f"Actual Fail    {tn:4d}      {fp:4d}")
print(f"Actual Pass    {fn:4d}      {tp:4d}")

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=True,
            xticklabels=['Predicted Fail', 'Predicted Pass'],
            yticklabels=['Actual Fail', 'Actual Pass'],
            annot_kws={"size": 16})
plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.ylabel('Actual Class', fontsize=12)
plt.xlabel('Predicted Class', fontsize=12)
plt.tight_layout()
plt.show()

Calculăm și afișăm metricile pe datele de antrenare.

In [ ]:
accuracy = accuracy_score(passed, y_pred)
precision = precision_score(passed, y_pred)
recall = recall_score(passed, y_pred)
f1 = f1_score(passed, y_pred)

print(f"{'=' * 50}")
print("PERFORMANCE METRICS")
print(f"{'=' * 50}")
print(f"True Positives (TP):   {tp}")
print(f"True Negatives (TN):   {tn}")
print(f"False Positives (FP):  {fp}")
print(f"False Negatives (FN):  {fn}")
print(f"\nAccuracy:  {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f} ({precision * 100:.2f}%)")
print(f"Recall:    {recall:.4f} ({recall * 100:.2f}%)")
print(f"F1-Score:  {f1:.4f}")

Prezentăm interpretarea metricilor.

In [ ]:
print(f"{'=' * 50}")
print("INTERPRETATION")
print(f"{'=' * 50}")
print(f"Accuracy:  {accuracy * 100:.2f}% of all predictions are correct")
print(f"Precision: {precision * 100:.2f}% of predicted passes are actual passes")
print(f"Recall:    {recall * 100:.2f}% of actual passes were correctly identified")
print(f"F1-Score:  {f1:.4f} (harmonic mean of precision and recall)")

Calculăm predicția pentru 4 ore de studiu.

In [ ]:
hours_4 = np.array([[4]])
prob_4 = model.predict_proba(hours_4)[0, 1]
pred_4 = model.predict(hours_4)[0]

print(f"{'=' * 50}")
print("PREDICTION FOR 4 HOURS OF STUDY")
print(f"{'=' * 50}")
print(f"Probability of passing: {prob_4:.4f} ({prob_4 * 100:.2f}%)")
print(f"Prediction: {'PASS' if pred_4 == 1 else 'FAIL'}")

Calculăm orele de studiu necesare pentru o șansă de 90% de a promova.

In [ ]:
hours_90 = (np.log(0.9 / 0.1) - b0) / b1
prob_90_check = model.predict_proba([[hours_90]])[0, 1]

print(f"{'=' * 50}")
print("MODEL ESTIMATE: 90% PASS PROBABILITY")
print(f"{'=' * 50}")
print(f"Study hours: {hours_90:.2f}")
print(f"Verification: {prob_90_check:.4f} ({prob_90_check * 100:.2f}%)")

Calculăm și afișăm probabilitățile de promovare pentru diferite ore de studiu.

In [ ]:
print(f"{'=' * 50}")
print("PASS PROBABILITIES FOR DIFFERENT STUDY HOURS")
print(f"{'=' * 50}")
for h in [2, 4, 6, 8]:
    prob = model.predict_proba([[h]])[0, 1]
    print(f"{h} hours: {prob:.4f} ({prob * 100:.2f}%)")

# Temă

**Sarcină:** Rezolvați și prezentați problemele următoare similar cu cele rezolvate la laborator.

Studenții de la specializarea Matematică informatică vor lucra în grupe de câte două persoane.

## Regresie Liniară
O cafenea estimează încasările zilnice din numărul de clienți. Fiecare observație reprezintă o zi; cheltuiala medie per client variază.

Date:

- Număr clienți: `[50, 60, 70, 80, 80, 100, 110, 110, 130, 140, 150, 160]`
- Încasări (lei): `[720, 940, 980, 1270, 1150, 1510, 1580, 1740, 1880, 2170, 2190, 2440]`

Sarcini:

1. Vizualizarea datelor pentru a înțelege natura relației.
2. Antrenarea unui model de regresie liniară simplă.
3. Calcularea metricilor de evaluare: MAE, MSE, RMSE și R².
4. Predicție pentru o zi cu 125 clienți folosind modelul antrenat.
5. Discuție despre adecvarea modelului liniar (underfitting) și validitatea predicției.

## Regresie logistică

Estimăm probabilitatea ca o livrare să întârzie din distanță și numărul de comenzi în așteptare la plasarea ei. Întârziere înseamnă o durată totală de peste 30 de minute.

Date:
- Distanță (km): `[2, 2, 2, 6, 6, 6, 10, 10, 10, 3, 3, 3, 5, 5, 5, 9, 9, 9]`
- Comenzi în așteptare: `[0, 3, 6, 0, 3, 6, 0, 3, 6, 1, 4, 7, 1, 4, 7, 1, 4, 7]`
- Întârziere (1=Da, 0=Nu): `[0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1]`

Sarcini:
1. Împărțire stratificată în seturi de antrenament (14 puncte) și testare (4 puncte), folosind `train_test_split` cu `test_size=4`, `stratify=y` și `random_state=42`.
2. Antrenare model de regresie logistică
3. Calcularea tuturor metricilor pentru seturile de date de antrenare și testare
4. Vizualizarea frontierei de decizie
5. Prezicerea probabilității de întârziere pentru o livrare la 6 km, cu 2 comenzi în așteptare
6. Discutarea influenței fiecărei caracteristici asupra predicției